In [70]:
from ipaddress import ip_address, IPv4Address, IPv6Address, IPv4Network, IPv6Network
import ipaddress
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()
fablib.verify_and_configure()

User: lngo@wcupa.edu bastion key is valid!
Configuration is valid
User: lngo@wcupa.edu bastion key is valid!
Configuration is valid
Please save the config!


## Docker Security

We want a 1-node Docker with 2 cores, 8GB of memory. 

In [71]:
# Get the resources helper
resources = fablib.get_resources()
resources.update()

# if you have multiple nodes and want adequate resources, you need find a suitable site
nodesReq = 1
coresReq = 2
ramReq = 4

# we scale up the requirements a bit to account for the potential of others joining the selected site. 
totalCoreAvail = nodesReq * coresReq * 1.2
totalRamAvail = nodesReq * ramReq * 1.2

usableSite = []
siteList = resources.get_site_names()
for site in siteList:
    cores = resources.get_core_available(site)
    ram = resources.get_ram_available(site)
    if cores >= totalCoreAvail and ram >= totalRamAvail:
        usableSite.append(site)

print(usableSite)

['FIU', 'UTAH', 'STAR', 'CERN', 'UCSD', 'NEWY', 'RUTG', 'CLEM', 'KANS', 'LOSA', 'MASS', 'MAX', 'TACC', 'SEAT', 'WASH', 'BRIST', 'EDUKY', 'EDC', 'AMST', 'PSC', 'MICH', 'GPN', 'HAWI', 'INDI', 'DALL', 'SRI', 'PRIN', 'GATECH', 'TOKY', 'ATLA', 'SALT', 'NCSA']


To avoid the scenario where all students joined the same site, the site selection is now random!

In [75]:
import random

usableSite = ['SRI','TOKY','BRIST'] # Current version of Docker rootless kit only works with inherent IPv4 sites. 
siteName = random.choice(usableSite)
#siteName = "CLEM"
sliceName = "Docker_NotRoot"
print(siteName)

network_name = 'ramnet'

slice = fablib.new_slice(name=sliceName)

# Network
net = slice.add_l2network(name=network_name, subnet=IPv4Network("192.168.1.0/24"))

for i in range(1, nodesReq + 1):
    node = slice.add_node(name=f"node{i}", 
                          site=siteName,
                          cores=coresReq,
                          ram=ramReq,
                          disk=30, 
                          image='default_ubuntu_24')
    iface = node.add_component(model='NIC_Basic', name='nic').get_interfaces()[0]
    iface.set_mode('config')
    net.add_interface(iface)

slice.submit()   
slice.wait_ssh()


Retry: 11, Time: 256 sec


ID,e2161f04-7c0c-4d80-b84a-80d0b5e9c5a5
Name,Docker_NotRoot
Lease Expiration (UTC),2026-04-15 23:58:11 +0000
Lease Start (UTC),2026-04-14 23:58:11 +0000
Project ID,8b6dfc51-02ad-4f00-a389-6e75d8b61a26
State,StableOK
Email,lngo@wcupa.edu
UserId,8eecd713-fa8f-4b3b-8883-1ff9b021fa53


ID,Name,Cores,RAM,Disk,Image,Image Type,Host,Site,Username,Management IP,State,Error,SSH Command,Public SSH Key File,Private SSH Key File
3513c2a3-4ed9-496a-85b5-ed091a2b238d,node1,2,4,100,default_ubuntu_24,qcow2,toky-w3.fabric-testbed.net,TOKY,ubuntu,133.69.160.195,Active,,ssh -i /home/fabric/.ssh/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@133.69.160.195,/home/fabric/.ssh/slice_key.pub,/home/fabric/.ssh/slice_key


ID,Name,Layer,Type,Site,Gateway,Subnet,State,Error
22d83465-cb5a-4ab1-ba92-56a03da48046,ramnet,L2,L2Bridge,TOKY,None,192.168.1.0/24,Active,


Name,Short Name,Node,Network,Bandwidth,VLAN,MAC,Physical Device,Device,Mode,IP Address,Numa Node,Switch Port
node1-nic-p1,p1,node1,ramnet,100,,06:10:C8:B2:1E:58,enp7s0,enp7s0,config,fe80::410:c8ff:feb2:1e58,4,HundredGigE0/0/0/9



Time to print interfaces 258 seconds


True

In [76]:
from ipaddress import IPv4Network

for i in range(1, nodesReq + 1):
    node = slice.get_node(name=f"node{i}")
    iface = node.get_interface(network_name=network_name)

    iface.ip_link_up()
    iface.ip_addr_add(
        addr=f"192.168.1.{i}",
        subnet=IPv4Network("192.168.1.0/24")
    )

In [77]:
import time

while True:
    time.sleep(10)
    slice.update()
    print("Slice state:", slice.get_state())
    print("Slice stable:", slice.isStable())
    if node.get_management_ip() != None:
        for node in slice.get_nodes():
            print("----", node.get_name(), "----")
            print("reservation state:", node.get_reservation_state())
            print("management ip:", node.get_management_ip())
            print("username:", node.get_username())
            print("error:", node.get_error_message())
            print(node.get_ssh_command())
        break

Slice state: StableOK
Slice stable: True
---- node1 ----
reservation state: Active
management ip: 133.69.160.195
username: ubuntu
error: 
ssh -i /home/fabric/.ssh/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@133.69.160.195


## Create Inventory File

This is a simple inventory file that reuse the `node` variable from the earlier cell directly (fewer calls to `slice`). 

Programmatically design the generation of `inventory.yml` based on this information. 

In [78]:
node = slice.get_node(name="node1")

docker_python, stderr = node.execute("which python3", quiet=True, output_file=f"{node.get_name()}-python.log");
inventory = f"""all:
  children:
    dockers:
      hosts:
        docker:
          ansible_host: "{node.get_management_ip()}"
          ansible_user: "{node.get_username()}"
          ansible_ssh_private_key_file: "{fablib.get_default_slice_key()["slice_private_key_file"]}"
          ansible_ssh_common_args: "-F /home/fabric/work/fabric_config/ssh_config"
          ansible_python_interpreter: "{docker_python.strip()}"
"""

from pathlib import Path
Path("playbook/inventory.yml").write_text(inventory)

343

## Playbook for Docker

- Primarily a declarative interpretation of [Docker's installation instruction](https://docs.docker.com/engine/install/ubuntu/#install-using-the-repository)
    - This playbook is different from the Docker Swarm playbook in that the registry is not enabled. 
- A few extra steps to enable IPv6 support for Docker (FABRIC related)

In [79]:
!ansible-playbook -i playbook/inventory.yml playbook/playbook-docker.yml


PLAY [Install Docker Engine on all nodes] **************************************

TASK [Gathering Facts] *********************************************************
ok: [docker]

TASK [Install prerequisite packages] *******************************************
ok: [docker]

TASK [Create Docker keyring directory] *****************************************
ok: [docker]

TASK [Download Docker GPG key] *************************************************
changed: [docker]

TASK [Add Docker apt repository] ***********************************************
changed: [docker]

TASK [Install Docker Engine] ***************************************************
changed: [docker]

TASK [Ensure docker group exists] **********************************************
ok: [docker]

TASK [Add ansible user to docker group] ****************************************
changed: [docker]

TASK [Ensure IPv4 forwarding is enabled] ***************************************
changed: [docker]

TASK [Ensure IPv6 forwarding is enabl

In [80]:
for node in slice.get_nodes():
    print(f"==== {node.get_name()} ====")
    stdout, stderr = node.execute("docker version", quiet=True);
    print(stdout)

==== node1 ====
Client: Docker Engine - Community
 Version:           29.4.0
 API version:       1.54
 Go version:        go1.26.1
 Git commit:        9d7ad9f
 Built:             Tue Apr  7 08:36:07 2026
 OS/Arch:           linux/amd64
 Context:           default

Server: Docker Engine - Community
 Engine:
  Version:          29.4.0
  API version:      1.54 (minimum version 1.40)
  Go version:       go1.26.1
  Git commit:       daa0cb7
  Built:            Tue Apr  7 08:36:07 2026
  OS/Arch:          linux/amd64
  Experimental:     false
 containerd:
  Version:          v2.2.2
  GitCommit:        301b2dac98f15c27117da5c8af12118a041a31d9
 runc:
  Version:          1.3.4
  GitCommit:        v1.3.4-0-gd6d73eb8
 docker-init:
  Version:          0.19.0
  GitCommit:        de40ad0



In [73]:
slice.delete()